# TrashScan — pelatihan pengklasifikasi sampah untuk BinGo

Notebook ini **mandiri**: tidak mengunduh dataset apa pun dan tidak butuh
`kaggle.json`. Seluruh peta label tertanam di dalamnya. Yang dibutuhkan hanya
dataset yang dilampirkan lewat panel **Input** di sebelah kanan.

Satu-satunya hal yang diunduh adalah bobot ImageNet MobileNetV3 (~4 MB) oleh
Keras, jadi **Internet harus On**.

## Sebelum Run All — lampirkan dataset

Klik **+ Add Input** di panel kanan, cari, lalu lampirkan sebanyak mungkin dari:

| Cari di kotak pencarian | Kelas yang dibawanya | Lisensi |
|---|---|---|
| `drinking waste classification` | `AluCan`, `Glass`, `PET`, `HDPEM` | CC0 |
| `trashnet` | `cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash` | MIT |
| `realwaste` | 9 kelas termasuk `Food Organics`, `Vegetation` | CC BY 4.0 |

Notebook **mengenali sendiri** mana yang terpasang dengan membaca nama folder
kelasnya, jadi slug dataset apa pun boleh — asal isinya satu folder per kelas.
Kalau ada yang belum dilampirkan, ia tetap jalan dan **menyebutkan konsekuensinya
secara eksplisit**, bukan diam-diam menghasilkan angka yang lebih rendah.

Setelan yang disarankan di panel kanan:

- **Accelerator: GPU T4 x2** (atau P100). Tanpa GPU, pelatihan ~10× lebih lambat.
- **Internet: On.** WAJIB. Bobot ImageNet MobileNetV3 diunduh Keras saat sel
  pelatihan pertama kali dijalankan (~4 MB). Tanpa internet, sel itu gagal
  dengan `URL fetch failure ... Tunnel connection failed`. Datasetnya sendiri
  tetap tidak diunduh — semuanya dari panel Input.
- **Persistence: Files only** kalau ingin artefak bertahan antar sesi.

## Yang dikeluarkan

Di `/kaggle/working/artifacts/`: `model_int8.tflite`, `labels.txt`,
`app_labels.json`, `metrics.json`, `report.md`. Unduh lewat tab **Output**.

In [ ]:
# @title Setup dan konfigurasi
SMOKE = False   # True = citra sintetis, ~3 menit, untuk memastikan notebook jalan

IMG          = 224     # MobileNetV3 diproyeksikan untuk 224
BATCH        = 64
EPOCHS_HEAD  = 6       # kepala saja, backbone beku
EPOCHS_FT    = 12      # 40 lapis terakhir dibuka
SEED         = 42
UNFREEZE     = 40

# 'imagenet' menuntut Internet: On di panel kanan. Setel None hanya untuk
# memeriksa notebooknya tanpa jaringan — akurasinya akan jatuh drastis, persis
# alasan CNN Thung & Yang hanya mencapai 27% pada poster aslinya.
WEIGHTS      = 'imagenet'

import os, sys, json, csv, time, shutil, random, subprocess
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IN   = Path('/kaggle/input')
WORK = Path('/kaggle/working')
OUT  = WORK / 'artifacts'; OUT.mkdir(parents=True, exist_ok=True)
DATA = WORK / 'unified'

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow {tf.__version__} | GPU: {[g.name for g in gpus] or "TIDAK ADA (akan lambat)"}')

# Mixed precision hanya bila ada GPU. Di CPU ia justru memperlambat, dan pada
# beberapa versi menghasilkan NaN pada lapisan normalisasi.
if gpus:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('mixed_float16 diaktifkan')

In [ ]:
# @title Peta label — tertanam, tidak mengambil dari mana pun
#
# Tiga lapis, dan pemisahannya menentukan apakah modelnya bisa dipasang:
#   KELAS LATIH  — apa yang bisa dibedakan model dari foto
#   MaterialType — satuan yang dipahami aplikasi BinGo
#   MaterialGrade— satuan yang menentukan harga di lapak
#
# Kelas latih sengaja TIDAK sama dengan MaterialType. Kardus dan kertas keduanya
# PAPER di mata aplikasi, tetapi jelas berbeda di mata kamera dan berbeda harga
# di lapak — jadi dilatih terpisah lalu digabung saat keluar sambil membawa
# grade KERTAS_KARDUS. Melebur keduanya sejak awal membuang sinyal gratis.

TRAIN_CLASSES = ['PET', 'HDPE', 'PLASTIC_OTHER', 'PAPER', 'CARDBOARD',
                 'METAL_CAN', 'METAL_OTHER', 'GLASS', 'ORGANIC', 'MIXED']

LABEL_ID = {
    'PET': 'Botol PET', 'HDPE': 'Plastik HDPE (galon, botol susu, sampo)',
    'PLASTIC_OTHER': 'Plastik lain', 'PAPER': 'Kertas', 'CARDBOARD': 'Kardus',
    'METAL_CAN': 'Kaleng aluminium', 'METAL_OTHER': 'Logam lain',
    'GLASS': 'Kaca', 'ORGANIC': 'Organik', 'MIXED': 'Campuran / residu',
}

# kelas latih -> (MaterialType, MaterialGrade|None, alasan)
TO_APP = {
    'PET': ('PET', None, 'bening dan berwarna beda harga; warnanya tidak dilabeli sumber mana pun'),
    'HDPE': ('HDPE', None, 'enum MaterialGrade belum punya grade HDPE'),
    'PLASTIC_OTHER': ('OTHER_PLASTIC', None, 'jenis resin tidak dilabeli'),
    'PAPER': ('PAPER', None, 'koran, arsip, dan duplex beda harga dan tercampur di sini'),
    'CARDBOARD': ('PAPER', 'KERTAS_KARDUS', 'kardus tidak ambigu'),
    'METAL_CAN': ('METAL', 'LOGAM_KALENG', 'kaleng minuman aluminium — tidak ambigu'),
    'METAL_OTHER': ('METAL', None, 'tembaga, besi, dan aluminium lembaran beda harga jauh'),
    'GLASS': ('GLASS', None, 'grade kaca satu-satunya di enum adalah KACA_BELING yang berarti '
                             'PECAHAN, sedangkan sumbernya botol utuh — tidak diklaim'),
    'ORGANIC': ('ORGANIC', None, ''),
    'MIXED': ('MIXED', None, ''),
}

UNREACHABLE_TYPES = ['PVC', 'LDPE', 'PS', 'PP']
ALL_GRADES_N = 18

# nama folder kelas per sumber -> (kelas latih, grade|None)
SOURCE_MAPS = {
    'drinking_waste': {
        'alucan': ('METAL_CAN', 'LOGAM_KALENG'), 'glass': ('GLASS', None),
        'pet': ('PET', None), 'hdpem': ('HDPE', None),
    },
    'trashnet': {
        'cardboard': ('CARDBOARD', 'KERTAS_KARDUS'), 'paper': ('PAPER', None),
        'glass': ('GLASS', None), 'metal': ('METAL_OTHER', None),
        'plastic': ('PLASTIC_OTHER', None), 'trash': ('MIXED', None),
    },
    'realwaste': {
        'cardboard': ('CARDBOARD', 'KERTAS_KARDUS'), 'paper': ('PAPER', None),
        'glass': ('GLASS', None), 'metal': ('METAL_OTHER', None),
        'plastic': ('PLASTIC_OTHER', None), 'food organics': ('ORGANIC', None),
        'vegetation': ('ORGANIC', None), 'textile trash': ('MIXED', None),
        'miscellaneous trash': ('MIXED', None),
    },
}

# Sidik jari untuk mengenali sumber dari nama-nama folder anaknya. Dipakai
# karena slug dataset di Kaggle berbeda-beda dan tidak bisa diandalkan.
FINGERPRINT = {
    'drinking_waste': {'alucan', 'hdpem'},
    'realwaste':      {'food organics', 'vegetation'},
    'trashnet':       {'cardboard', 'trash', 'plastic'},
}

SOURCE_META = {
    'drinking_waste': {'name': 'Drinking Waste Classification', 'license': 'CC0 1.0',
                       'attribution': 'Arkadiy Serezhkin, Drinking Waste Classification, Kaggle (CC0).'},
    'trashnet': {'name': 'TrashNet', 'license': 'MIT',
                 'attribution': 'Gary Thung & Mindy Yang, TrashNet, Stanford CS229 (2016), MIT.'},
    'realwaste': {'name': 'RealWaste', 'license': 'CC BY 4.0',
                  'attribution': 'Single, S., Iranmanesh, S., & Raad, R. (2023). RealWaste. '
                                 'UCI Machine Learning Repository. CC BY 4.0.'},
}

REACHABLE_GRADES = sorted({g for m in SOURCE_MAPS.values() for (_, g) in m.values() if g})
N = len(TRAIN_CLASSES)
CLS_IDX = {c: i for i, c in enumerate(TRAIN_CLASSES)}

print(f'{N} kelas latih -> {len({TO_APP[c][0] for c in TRAIN_CLASSES})} MaterialType '
      f'-> {len(REACHABLE_GRADES)} dari {ALL_GRADES_N} grade')
print('MaterialType tidak tercapai:', ', '.join(UNREACHABLE_TYPES))

In [ ]:
# @title Gaya grafik
# Palet lolos uji keterpisahan buta warna pada semua pasangan. Satu seri = satu
# warna dan tanpa legenda; judulnya sudah menyebut apa yang diukur.
SURFACE, INK, INK_MUTED = '#fcfcfb', '#0b0b0b', '#52514e'
SERIES = ['#2a78d6', '#eb6834', '#1baf7a']
SEQ    = ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#2a78d6', '#256abf', '#1c5cab', '#184f95']
from matplotlib.colors import LinearSegmentedColormap
CMAP = LinearSegmentedColormap.from_list('bingo', SEQ)

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE,
    'axes.edgecolor': '#d8d7d3', 'axes.labelcolor': INK_MUTED,
    'axes.titlecolor': INK, 'axes.titleweight': 'bold', 'axes.titlesize': 12,
    'xtick.color': INK_MUTED, 'ytick.color': INK_MUTED,
    'axes.grid': True, 'grid.color': '#ebeae6', 'grid.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'figure.dpi': 120,
})
print('gaya grafik siap')

## 1 — Menemukan dataset yang terpasang

Notebook menelusuri `/kaggle/input` dan mengenali sumber dari **nama folder
kelasnya**, bukan dari slug datasetnya. Artinya mirror mana pun boleh dipakai
asal strukturnya satu folder per kelas.

Kalau sebuah sumber tidak terpasang, konsekuensinya disebutkan terang-terangan —
bukan dibiarkan muncul sebagai angka yang diam-diam lebih rendah.

In [ ]:
# @title Telusuri /kaggle/input dan kenali sumbernya
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def classify_dir(d: Path):
    # Kenali sumber dari himpunan nama folder anaknya. None bila bukan akar kelas.
    kids = {p.name.strip().lower() for p in d.iterdir() if p.is_dir()}
    if not kids:
        return None
    for src, fp in FINGERPRINT.items():
        if fp <= kids:
            return src
    return None

def discover():
    # Kembalikan {sumber: Path akar kelas}. Bila satu sumber muncul di beberapa
    # tempat (dataset ganda atau nested), ambil yang citranya paling banyak.
    found = {}
    if not IN.exists():
        return found
    stack = [IN]
    while stack:
        d = stack.pop()
        try:
            kids = [p for p in d.iterdir() if p.is_dir()]
        except (PermissionError, OSError):
            continue
        src = classify_dir(d)
        if src:
            n = sum(1 for _ in d.rglob('*') if _.suffix.lower() in IMG_EXT)
            if src not in found or n > found[src][1]:
                found[src] = (d, n)
            continue          # jangan turun lagi; ini sudah akar kelas
        stack.extend(kids)
    return {k: v[0] for k, v in found.items()}

if SMOKE:
    roots = {}
    print('mode SMOKE — penelusuran dilewati')
else:
    roots = discover()
    print('Dataset terdeteksi:')
    for s in ('drinking_waste', 'trashnet', 'realwaste'):
        if s in roots:
            print(f'  [OK]     {SOURCE_META[s]["name"]:32s} {roots[s]}')
        else:
            print(f'  [KURANG] {SOURCE_META[s]["name"]:32s} — belum dilampirkan')

    if not roots:
        raise SystemExit(
            'Tidak ada dataset yang dikenali di /kaggle/input.\n'
            'Klik "+ Add Input" di panel kanan lalu lampirkan minimal satu dari:\n'
            '  drinking waste classification / trashnet / realwaste\n'
            'Atau setel SMOKE = True untuk memeriksa notebooknya saja.')

    missing = {'drinking_waste', 'trashnet', 'realwaste'} - set(roots)
    if 'drinking_waste' in missing:
        print('\n>> PERINGATAN: tanpa Drinking Waste, kelas HDPE dan METAL_CAN kehilangan')
        print('   hampir seluruh contohnya, dan PET kehilangan sumber terbesarnya.')
        print('   Ini WAJIB disebut bila angka hasilnya dikutip.')
    if 'realwaste' in missing:
        print('\n>> PERINGATAN: tanpa RealWaste, kelas ORGANIC hilang sama sekali dan')
        print('   seluruh data berasal dari foto beralas posterboard putih. Angka')
        print('   dalam-sumber akan tampak tinggi tetapi tidak berarti di lapangan.')

In [ ]:
# @title Bangun manifest dan buang duplikat
from PIL import Image

def dhash_int(path, size=8):
    # Difference hash. Dipakai dua kali: membuang duplikat lintas-sumber, dan
    # nanti memisahkan train/val/test per klaster supaya foto objek fisik yang
    # sama tidak pernah jatuh di train dan test sekaligus.
    try:
        im = Image.open(path).convert('L').resize((size + 1, size), Image.LANCZOS)
    except Exception:
        return None
    px = np.asarray(im, dtype=np.int16)
    bits = (px[:, 1:] > px[:, :-1]).flatten()
    v = 0
    for b in bits:
        v = (v << 1) | int(b)
    return v

def cluster(hashes, threshold=5):
    # Kelompokkan foto yang nyaris sama. Bucketing 16 bit teratas memangkas
    # perbandingan dari O(n^2) penuh jadi sekadar cepat pada puluhan ribu citra.
    n = len(hashes); cid = [-1] * n
    buckets = defaultdict(list)
    for i, h in enumerate(hashes):
        buckets[h >> 48].append(i)
    nxt = 0
    for i in range(n):
        if cid[i] != -1:
            continue
        cid[i] = nxt
        for j in buckets[hashes[i] >> 48]:
            if j > i and cid[j] == -1 and bin(hashes[i] ^ hashes[j]).count('1') <= threshold:
                cid[j] = nxt
        nxt += 1
    return cid

rows = []
if SMOKE:
    # Warna diturunkan dari TRAIN_CLASSES supaya mode uji ikut berubah sendiri
    # ketika daftar kelas berubah. Mengetik nama kelas manual membuat manifest
    # sintetis diam-diam kosong begitu taksonominya diganti.
    base = [(150,190,225),(225,228,230),(120,175,150),(228,225,214),(172,132,92),
            (196,168,120),(168,170,175),(172,202,206),(120,158,104),(108,100,94)]
    rng = np.random.default_rng(SEED)
    for mi, m in enumerate(TRAIN_CLASSES):
        d = DATA / m; d.mkdir(parents=True, exist_ok=True)
        for i in range(48):
            src = ['trashnet', 'realwaste', 'drinking_waste'][i % 3]
            a = np.clip(rng.normal(base[mi % len(base)], 24, (72, 72, 3)), 0, 255).astype(np.uint8)
            p = d / f'{src}__{i}.jpg'
            Image.fromarray(a).save(p, quality=88)
            rows.append({'path': str(p), 'source': src, 'cls': m, 'grade': '',
                         'dhash': mi * 1000 + i, 'cluster': mi * 1000 + i})
    print(f'mode uji: {len(rows)} citra sintetis, {N} kelas')
else:
    unknown = Counter()
    for src, root in roots.items():
        m = SOURCE_MAPS[src]
        n_before = len(rows)
        for d in sorted(p for p in root.iterdir() if p.is_dir()):
            key = d.name.strip().lower()
            if key not in m:
                unknown[f'{src}/{d.name}'] += 1
                continue
            cls, grade = m[key]
            for f in d.rglob('*'):
                if f.suffix.lower() in IMG_EXT and f.is_file():
                    rows.append({'path': str(f), 'source': src, 'cls': cls,
                                 'grade': grade or ''})
        print(f'{SOURCE_META[src]["name"]:32s} {len(rows) - n_before:>6,} citra')
    if unknown:
        print('\nFolder yang tidak dikenal (dilewati):', dict(unknown))

    print(f'\n{len(rows):,} citra terpetakan. Menghitung hash duplikat…')
    keep, hashes = [], []
    for i, r in enumerate(rows):
        if i and i % 2000 == 0:
            print(f'  {i:,}/{len(rows):,}')
        h = dhash_int(r['path'])
        if h is None:
            continue
        r['dhash'] = h; hashes.append(h); keep.append(r)
    rows = keep
    for r, c in zip(rows, cluster(hashes)):
        r['cluster'] = c

    seen, deduped = set(), []
    for r in rows:
        if r['cluster'] in seen:
            continue
        seen.add(r['cluster']); deduped.append(r)
    print(f'{len(rows) - len(deduped):,} dibuang sebagai duplikat; {len(deduped):,} tersisa')
    rows = deduped

man = rows
counts = Counter(r['cls'] for r in man)
print('\nper kelas :', {c: counts.get(c, 0) for c in TRAIN_CLASSES})
print('per sumber:', dict(Counter(r['source'] for r in man)))

In [ ]:
# @title Sebaran kelas
order = [c for c in TRAIN_CLASSES if counts.get(c)]
vals  = [counts[c] for c in order]

fig, ax = plt.subplots(figsize=(7, 0.42 * len(order) + 1.2))
y = np.arange(len(order))
ax.barh(y, vals, color=SERIES[0], height=0.62)
ax.set_yticks(y); ax.set_yticklabels([LABEL_ID[c] for c in order])
ax.invert_yaxis(); ax.set_xlabel('jumlah citra')
ax.set_title('Sebaran citra per kelas latih')
for yi, v in zip(y, vals):
    ax.text(v + max(vals) * 0.012, yi, f'{v:,}', va='center', color=INK_MUTED, fontsize=9)
ax.set_xlim(0, max(vals) * 1.12)
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

imb = max(vals) / max(1, min(vals))
print(f'Ketimpangan terbesar : {imb:.1f}x  ({order[int(np.argmax(vals))]} vs {order[int(np.argmin(vals))]})')
print('Karena itu yang dilaporkan macro-F1, bukan akurasi — akurasi tunggal')
print('menyembunyikan kelas yang gagal total.')
if len(order) < N:
    print(f'\nKelas TANPA data sama sekali: {", ".join(c for c in TRAIN_CLASSES if not counts.get(c))}')

## 2 — Pemisahan train/val/test

Dipisah **per klaster foto**, bukan per foto. Tanpa ini dua foto objek fisik yang
sama bisa jatuh di train dan test sekaligus, dan akurasinya naik semu. Ini
kesalahan yang paling sering menggelembungkan angka pada laporan klasifikasi
sampah.

Rasio 70/13/17 mengikuti Thung & Yang (2016) supaya angkanya bisa disandingkan
langsung dengan **75%** yang mereka laporkan. Kalau angkamu lebih rendah,
kemungkinan besar karena kamu memisahkan dengan benar dan mereka tidak.

In [ ]:
# @title Split per klaster, proporsi kelas dijaga
by_cluster = defaultdict(list)
for i, r in enumerate(man):
    by_cluster[r['cluster']].append(i)

# Klaster diurutkan per kelas dominannya lalu dibagi bergiliran, sehingga
# proporsi kelas terjaga tanpa pernah memecah satu klaster ke dua split.
clusters_by_cls = defaultdict(list)
for cid, idx in by_cluster.items():
    dom = Counter(man[i]['cls'] for i in idx).most_common(1)[0][0]
    clusters_by_cls[dom].append(cid)

rng = random.Random(SEED)
tr_i, va_i, te_i = [], [], []
for cls, cids in clusters_by_cls.items():
    rng.shuffle(cids)
    n = len(cids); n_tr = int(n * 0.70); n_va = int(n * 0.13)
    for j, cid in enumerate(cids):
        bucket = tr_i if j < n_tr else (va_i if j < n_tr + n_va else te_i)
        bucket.extend(by_cluster[cid])

print(f'klaster : {len(by_cluster):,}')
print(f'train {len(tr_i):,} | val {len(va_i):,} | test {len(te_i):,}')
for name, idx in (('train', tr_i), ('val', va_i), ('test', te_i)):
    c = Counter(man[i]['cls'] for i in idx)
    print(f'  {name:5s}', {k: c.get(k, 0) for k in TRAIN_CLASSES})

In [ ]:
# @title Pipeline data
AUTOTUNE = tf.data.AUTOTUNE

paths  = np.array([r['path'] for r in man])
labels = np.array([CLS_IDX[r['cls']] for r in man], dtype=np.int32)

def decode(path, label, training):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG, IMG))
    if training:
        # Augmentasi sengaja konservatif. Flip vertikal dan rotasi besar tidak
        # dipakai: botol dan kaleng punya orientasi alami, dan memutarnya justru
        # mengajarkan invariansi yang tidak pernah ditemui di lapangan.
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.22)
        img = tf.image.random_contrast(img, 0.82, 1.18)
        img = tf.image.random_saturation(img, 0.82, 1.18)
        img = tf.clip_by_value(img, 0.0, 255.0)
    img.set_shape([IMG, IMG, 3])
    return img, label

def make_ds(idx, training):
    ds = tf.data.Dataset.from_tensor_slices((paths[idx], labels[idx]))
    if training:
        ds = ds.shuffle(min(len(idx), 4096), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, l: decode(p, l, training), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

ds_tr, ds_va, ds_te = make_ds(tr_i, True), make_ds(va_i, False), make_ds(te_i, False)

# Bobot kelas menangani ketimpangan tanpa membuang data seperti undersampling.
cnt = Counter(labels[tr_i])
total = sum(cnt.values())
class_weight = {c: total / (len(cnt) * cnt[c]) for c in cnt}
print('batch train:', len(list(ds_tr.take(1))), '| bobot kelas:',
      {TRAIN_CLASSES[k]: round(v, 2) for k, v in sorted(class_weight.items())})

## 3 — Model

**MobileNetV3-Small** dengan bobot ImageNet. Dipilih karena harus muat dan cepat
di ponsel kelas bawah, bukan karena ia arsitektur terbaik — dan itu memang bukan.

Dua tahap: kepala dilatih dulu dengan backbone dibekukan, lalu 40 lapis terakhir
dibuka pada laju belajar kecil. Membuka seluruh backbone sejak awal akan merusak
fitur ImageNet sebelum kepalanya sempat masuk akal.

In [ ]:
# @title Latih
def build():
    try:
        base = tf.keras.applications.MobileNetV3Small(
            input_shape=(IMG, IMG, 3), include_top=False, weights=WEIGHTS,
            include_preprocessing=True, pooling='avg')
    except Exception as e:
        # Kegagalan bawaan Keras di sini berbunyi "URL fetch failure ... Tunnel
        # connection failed", yang tidak memberitahu apa pun tentang apa yang
        # harus dilakukan. Diterjemahkan supaya tidak menghabiskan waktu orang.
        raise SystemExit(
            f'Gagal memuat bobot ImageNet: {e}\n\n'
            'Penyebab paling lazim: Internet masih Off.\n'
            'Buka panel kanan -> Settings -> Internet -> On, lalu jalankan ulang.\n'
            'Bobotnya hanya ~4 MB dan diunduh sekali; dataset tetap dari panel Input.\n\n'
            'Kalau memang ingin jalan tanpa internet, setel WEIGHTS = None di sel '
            'konfigurasi — tetapi akurasinya akan jatuh drastis dan angkanya tidak '
            'layak dikutip.') from e
    base.trainable = False
    x = base.output
    x = tf.keras.layers.Dropout(0.3)(x)
    # dtype float32 pada lapisan akhir wajib saat mixed precision: softmax di
    # float16 kehilangan presisi justru di rentang yang menentukan kalibrasi.
    out = tf.keras.layers.Dense(N, activation='softmax', dtype='float32')(x)
    return tf.keras.Model(base.input, out), base

model, base = build()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

cb = [tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4,
                                       restore_best_weights=True, verbose=1)]
print('=== Tahap 1: kepala saja ===')
h1 = model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS_HEAD,
               class_weight=class_weight, callbacks=cb, verbose=2)

print('\n=== Tahap 2: fine-tune 40 lapis terakhir ===')
base.trainable = True
for l in base.layers[:-UNFREEZE]:
    l.trainable = False
# BatchNorm dibiarkan beku. Statistik batch pada dataset kecil dan timpang
# bergerak liar, dan itu satu-satunya penyebab paling sering fine-tune justru
# memperburuk hasil.
for l in base.layers:
    if isinstance(l, tf.keras.layers.BatchNormalization):
        l.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h2 = model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS_FT,
               class_weight=class_weight, callbacks=cb, verbose=2)

In [ ]:
# @title Kurva belajar
acc  = h1.history['accuracy'] + h2.history['accuracy']
vacc = h1.history['val_accuracy'] + h2.history['val_accuracy']
split_at = len(h1.history['accuracy'])

fig, ax = plt.subplots(figsize=(7, 3.4))
e = np.arange(1, len(acc) + 1)
ax.plot(e, acc,  color=SERIES[0], lw=2, label='train')
ax.plot(e, vacc, color=SERIES[1], lw=2, label='validation')
ax.axvline(split_at + 0.5, color=INK_MUTED, lw=1, ls='--')
ax.text(split_at + 0.7, min(acc + vacc), ' fine-tune dimulai', color=INK_MUTED, fontsize=9)
ax.set_xlabel('epoch'); ax.set_ylabel('akurasi'); ax.set_ylim(0, 1)
ax.set_title('Akurasi per epoch')
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 4 — Evaluasi

In [ ]:
# @title Macro-F1 dan tabel per kelas
def predict(ds):
    p = model.predict(ds, verbose=0)
    return p

probs_te = predict(ds_te)
y_te = labels[te_i]
pred = probs_te.argmax(1)

def per_class_prf(y, p, n):
    out = {}
    for c in range(n):
        tp = int(((p == c) & (y == c)).sum())
        fp = int(((p == c) & (y != c)).sum())
        fn = int(((p != c) & (y == c)).sum())
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec  = tp / (tp + fn) if tp + fn else 0.0
        f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        out[c] = {'precision': prec, 'recall': rec, 'f1': f1, 'support': int((y == c).sum())}
    return out

def macro_f1(y, p, n):
    d = per_class_prf(y, p, n)
    seen = [c for c in range(n) if d[c]['support'] > 0]
    return sum(d[c]['f1'] for c in seen) / max(1, len(seen))

prf = per_class_prf(y_te, pred, N)
acc_te = float((pred == y_te).mean())
mf1 = macro_f1(y_te, pred, N)

print(f'Akurasi test : {acc_te:.4f}')
print(f'Macro-F1     : {mf1:.4f}\n')
print(f'{"Kelas":42s} {"P":>6s} {"R":>6s} {"F1":>6s} {"n":>6s}')
for c in range(N):
    v = prf[c]
    if v['support'] == 0:
        continue
    print(f'{LABEL_ID[TRAIN_CLASSES[c]]:42s} {v["precision"]:6.3f} {v["recall"]:6.3f} '
          f'{v["f1"]:6.3f} {v["support"]:6d}')

# Dua baseline. Kalau model tidak mengalahkan keduanya dengan selisih jelas,
# ia belum layak dipasang.
maj = Counter(labels[tr_i]).most_common(1)[0][0]
print(f'\nBaseline kelas mayoritas: akurasi {float((y_te == maj).mean()):.4f}, '
      f'macro-F1 {macro_f1(y_te, np.full_like(y_te, maj), N):.4f}')

In [ ]:
# @title Confusion matrix
cm = np.zeros((N, N), dtype=int)
for t, p in zip(y_te, pred):
    cm[t, p] += 1
row = cm.sum(1, keepdims=True)
cmn = np.divide(cm, np.maximum(row, 1))

fig, ax = plt.subplots(figsize=(8.5, 7.5))
im = ax.imshow(cmn, cmap=CMAP, vmin=0, vmax=1)
ax.set_xticks(range(N)); ax.set_xticklabels(TRAIN_CLASSES, rotation=45, ha='right')
ax.set_yticks(range(N)); ax.set_yticklabels([LABEL_ID[c] for c in TRAIN_CLASSES])
ax.set_xlabel('prediksi'); ax.set_ylabel('sebenarnya')
ax.set_title('Confusion matrix (proporsi per baris)')
ax.grid(False)
# Setiap sel diberi angka: warna saja tidak cukup terbaca bagi sebagian pembaca.
for i in range(N):
    for j in range(N):
        if cm[i, j]:
            ax.text(j, i, f'{cmn[i, j]:.2f}\n{cm[i, j]}', ha='center', va='center',
                    fontsize=7, color='white' if cmn[i, j] > 0.55 else INK)
fig.colorbar(im, ax=ax, shrink=0.8, label='proporsi')
plt.tight_layout(); plt.show()

## 5 — Kalibrasi

Akurasi menjawab "seberapa sering benar". Kalibrasi menjawab pertanyaan yang
berbeda dan justru lebih penting bagi aplikasi ini: **ketika model bilang yakin
80%, apakah ia benar 80% kali?** Kalau tidak, ambang abstain di aplikasi tidak
punya arti sama sekali.

Temperature scaling memasang satu parameter pada data **validation**, bukan test.
Ia tidak mengubah urutan prediksi, jadi akurasi tetap; yang berubah hanya sebaran
keyakinannya.

In [ ]:
# @title Temperature scaling + reliability diagram
probs_va = predict(ds_va)
y_va = labels[va_i]
logits_va = np.log(np.clip(probs_va, 1e-12, 1))

def ece(p, y, bins=15):
    conf, pred_ = p.max(1), p.argmax(1)
    correct = (pred_ == y).astype(float)
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def softmax_T(logits, T):
    z = logits / T
    z = z - z.max(1, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(1, keepdims=True)

best_t, best_e = 1.0, ece(probs_va, y_va)
e_before = best_e
for T in np.arange(0.5, 5.01, 0.05):
    e = ece(softmax_T(logits_va, T), y_va)
    if e < best_e:
        best_t, best_e = float(T), e
e_after = best_e

logits_te = np.log(np.clip(probs_te, 1e-12, 1))
probs_te_cal = softmax_T(logits_te, best_t)
print(f'ECE validation sebelum : {e_before:.4f}')
print(f'ECE validation sesudah : {e_after:.4f}  (T = {best_t:.2f})')
print(f'ECE test sesudah       : {ece(probs_te_cal, y_te):.4f}')

fig, ax = plt.subplots(figsize=(5.4, 5))
edges = np.linspace(0, 1, 11)
for probs, color, label in ((probs_te, SERIES[1], 'sebelum'), (probs_te_cal, SERIES[0], 'sesudah')):
    conf, pr = probs.max(1), probs.argmax(1)
    corr = (pr == y_te).astype(float)
    xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() >= 5:
            xs.append(conf[m].mean()); ys.append(corr[m].mean())
    ax.plot(xs, ys, 'o-', color=color, lw=2, ms=7, label=label)
ax.plot([0, 1], [0, 1], ls='--', color=INK_MUTED, lw=1)
ax.set_xlabel('keyakinan model'); ax.set_ylabel('akurasi sebenarnya')
ax.set_title('Reliability diagram (test)')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 6 — Perilaku abstain

Aplikasi menahan jawaban ketika model tidak yakin, lalu meminta pengguna memilih
material sendiri atau memotret kode resin. Tabel inilah yang menetapkan
ambangnya — bukan tebakan.

Dua garis, dua besaran, **satu sumbu** — keduanya berskala 0–1, jadi tidak perlu
sumbu ganda. Titik yang dicari adalah tempat macro-F1 sudah tinggi sementara
cakupan belum runtuh.

In [ ]:
# @title Kurva abstain
ths = np.arange(0.0, 0.96, 0.05)
cov, f1s = [], []
conf_te = probs_te_cal.max(1)
for th in ths:
    m = conf_te >= th
    cov.append(float(m.mean()))
    f1s.append(macro_f1(y_te[m], pred[m], N) if m.sum() >= N else 0.0)

# Ambang dipilih sebagai cakupan tertinggi yang masih menjaga macro-F1 dalam
# 2 poin dari puncaknya — bukan yang F1-nya tertinggi, karena itu biasanya
# menjawab terlalu sedikit pertanyaan untuk berguna.
peak = max(f1s)
ok = [i for i, f in enumerate(f1s) if f >= peak - 0.02]
pick = min(ok) if ok else 0
TH = float(ths[pick])

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(ths, cov, color=SERIES[0], lw=2, label='cakupan')
ax.plot(ths, f1s, color=SERIES[1], lw=2, label='macro-F1')
ax.axvline(TH, color=SERIES[2], lw=2, ls='--')
ax.text(TH + 0.01, 0.05, f' ambang {TH:.2f}', color=SERIES[2], fontsize=9, fontweight='bold')
ax.set_xlabel('ambang keyakinan'); ax.set_ylim(0, 1.02)
ax.set_title('Cakupan versus macro-F1 pada berbagai ambang')
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

print(f'Ambang terpilih {TH:.2f}: menjawab {cov[pick]:.1%} kasus, macro-F1 {f1s[pick]:.3f}')
print(f'Sisanya {1 - cov[pick]:.1%} dilempar ke koreksi manual atau kode resin.')

## 7 — Uji lintas-dataset

Ini bagian yang paling jujur, dan yang paling berguna dihadapkan ke juri justru
karena ia tidak memihak.

TrashNet difoto dengan objek diletakkan **di atas posterboard putih**. RealWaste
difoto di **titik penerimaan TPA sungguhan**. Model yang belajar dari satu sumber
lalu diuji pada sumber lain akan turun angkanya — **seberapa jauh turunnya adalah
perkiraan terbaik tentang apa yang terjadi saat model bertemu foto ponsel
pemulung di lapangan.**

Angka dalam-sumber yang tinggi tanpa angka lintas-sumber adalah klaim yang belum
diuji.

In [ ]:
# @title Latih per sumber, uji silang
srcs = sorted({r['source'] for r in man})
res = {}

if len(srcs) < 2:
    print('Hanya satu sumber terpasang — uji lintas-dataset dilewati.')
    print('Ini justru angka terkuat untuk sidang; lampirkan minimal dua dataset.')
else:
    EP = 1 if SMOKE else 4
    for a in srcs:
        idx_a = [i for i in tr_i if man[i]['source'] == a]
        if len(idx_a) < BATCH:
            print(f'{a}: terlalu sedikit contoh, dilewati'); continue
        m2, b2 = build()
        m2.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        m2.fit(make_ds(idx_a, True), epochs=EP, verbose=0)
        for b in srcs:
            idx_b = [i for i in te_i if man[i]['source'] == b]
            if len(idx_b) < N:
                continue
            pb = m2.predict(make_ds(idx_b, False), verbose=0).argmax(1)
            res[(a, b)] = macro_f1(labels[idx_b], pb, N)
            print(f'  latih {a:16s} -> uji {b:16s} macro-F1 {res[(a, b)]:.3f}')
        del m2, b2
        tf.keras.backend.clear_session()

    if res:
        fig, ax = plt.subplots(figsize=(1.6 * len(srcs) + 3, 1.4 * len(srcs) + 2.4))
        M = np.full((len(srcs), len(srcs)), np.nan)
        for (a, b), v in res.items():
            M[srcs.index(a), srcs.index(b)] = v
        im = ax.imshow(M, cmap=CMAP, vmin=0, vmax=1)
        ax.set_xticks(range(len(srcs))); ax.set_xticklabels(srcs, rotation=30, ha='right')
        ax.set_yticks(range(len(srcs))); ax.set_yticklabels(srcs)
        ax.set_xlabel('diuji pada'); ax.set_ylabel('dilatih pada')
        ax.set_title('Macro-F1 lintas-dataset'); ax.grid(False)
        for i in range(len(srcs)):
            for j in range(len(srcs)):
                if not np.isnan(M[i, j]):
                    ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center',
                            fontsize=10, color='white' if M[i, j] > 0.55 else INK)
        fig.colorbar(im, ax=ax, shrink=0.8, label='macro-F1')
        plt.tight_layout(); plt.show()

        diag = [res[(s, s)] for s in srcs if (s, s) in res]
        off  = [v for (a, b), v in res.items() if a != b]
        if diag and off:
            print(f'\nRata-rata dalam-sumber : {np.mean(diag):.3f}')
            print(f'Rata-rata lintas-sumber: {np.mean(off):.3f}')
            print(f'Selisih                : {np.mean(diag) - np.mean(off):.3f}')
            print('\nSelisih itulah angka yang harus dibawa ke sidang. Ia perkiraan')
            print('terbaik tentang penurunan yang akan terjadi di lapangan.')

## 8 — Ekspor ke TFLite int8

In [ ]:
# @title Kuantisasi, ukuran, latensi
sm = WORK / '_saved_model'
if sm.exists():
    shutil.rmtree(sm)
model.export(str(sm))

conv = tf.lite.TFLiteConverter.from_saved_model(str(sm))
conv.optimizations = [tf.lite.Optimize.DEFAULT]

# Dataset representatif WAJIB berasal dari data latih sungguhan. Kuantisasi
# menyetel rentang tiap tensor dari contoh ini; memakai noise acak menghasilkan
# rentang yang salah dan akurasi anjlok tanpa satu pun galat.
rep_idx = tr_i[:200]
def rep():
    for i in rep_idx:
        img = tf.io.decode_image(tf.io.read_file(man[i]['path']), channels=3,
                                 expand_animations=False)
        img = tf.image.resize(img, (IMG, IMG))
        yield [tf.expand_dims(tf.cast(img, tf.float32), 0).numpy()]
conv.representative_dataset = rep

tfl = conv.convert()
(OUT / 'model_int8.tflite').write_bytes(tfl)
(OUT / 'labels.txt').write_text('\n'.join(TRAIN_CLASSES) + '\n')
size_kb = len(tfl) / 1024

# Jembatan ke enum aplikasi. Tanpa berkas ini keluaran model tidak berarti apa
# pun bagi BinGo: CARDBOARD dan METAL_CAN bukan nilai MaterialType yang sah.
(OUT / 'app_labels.json').write_text(json.dumps({
    'trainClasses': TRAIN_CLASSES,
    'toApp': {c: {'materialType': TO_APP[c][0], 'grade': TO_APP[c][1],
                  'note': TO_APP[c][2]} for c in TRAIN_CLASSES},
    'unreachableMaterialTypes': UNREACHABLE_TYPES,
    'reachableGrades': REACHABLE_GRADES,
    'abstainThreshold': TH,
    'temperature': best_t,
}, indent=2, ensure_ascii=False))

itp = tf.lite.Interpreter(model_content=tfl); itp.allocate_tensors()
di, do = itp.get_input_details()[0], itp.get_output_details()[0]
z = np.zeros(di['shape'], dtype=di['dtype'])
for _ in range(5):
    itp.set_tensor(di['index'], z); itp.invoke()
t0 = time.time()
for _ in range(30):
    itp.set_tensor(di['index'], z); itp.invoke()
lat = (time.time() - t0) / 30 * 1000
shutil.rmtree(sm, ignore_errors=True)

print(f'model_int8.tflite  {size_kb:.0f} KB  |  {lat:.1f} ms per citra (CPU Kaggle)')
print('UKUR ULANG latensi di perangkat Android target sebelum angkanya dikutip.')

In [ ]:
# @title Simpan metrik dan laporan
present = sorted({r['source'] for r in man})
missing = [s for s in ('drinking_waste', 'trashnet', 'realwaste') if s not in present]

metrics = {
    'dataset': {
        'images': len(man), 'classes': TRAIN_CLASSES,
        'per_class': {c: counts.get(c, 0) for c in TRAIN_CLASSES},
        'per_source': dict(Counter(r['source'] for r in man)),
        'clusters': len(by_cluster),
        'sources_present': present, 'sources_missing': missing,
        'split': {'train': len(tr_i), 'val': len(va_i), 'test': len(te_i),
                  'policy': 'per klaster near-duplicate, 70/13/17 (Thung & Yang 2016)'},
        'licenses': {s: SOURCE_META[s] for s in present},
    },
    'model': {'backbone': 'MobileNetV3-Small', 'input': [IMG, IMG, 3],
              'epochs_head': EPOCHS_HEAD, 'epochs_finetune': EPOCHS_FT,
              'unfrozen_layers': UNFREEZE, 'weights': WEIGHTS or 'random-init'},
    'test': {'accuracy': acc_te, 'macro_f1': mf1,
             'per_class': {TRAIN_CLASSES[c]: v for c, v in prf.items()}},
    'calibration': {'temperature': best_t, 'ece_before': e_before, 'ece_after': e_after},
    'abstain': {'threshold': TH, 'coverage': cov[pick], 'macro_f1_at_threshold': f1s[pick]},
    'cross_dataset': {f'{a}->{b}': v for (a, b), v in res.items()},
    'edge': {'tflite_int8_kb': size_kb, 'latency_ms_kaggle_cpu': lat},
    'coverage_note': {'grades_reachable': REACHABLE_GRADES, 'grades_total': ALL_GRADES_N,
                      'material_types_unreachable': UNREACHABLE_TYPES},
    'smoke_mode': SMOKE,
}
(OUT / 'metrics.json').write_text(json.dumps(metrics, indent=2, ensure_ascii=False))

L = ['# Hasil pelatihan TrashScan', '']
if SMOKE:
    L += ['> **MODE SMOKE — citra sintetis. Angka di bawah acak dan TIDAK BOLEH dikutip.**', '']
if not WEIGHTS:
    L += ['> **Tanpa bobot ImageNet (WEIGHTS = None). Angka di bawah jauh di bawah '
          'kemampuan sebenarnya dan tidak layak dikutip.**', '']
L += [f'{len(man):,} citra, {N} kelas latih, dari {len(present)} dataset publik.',
      f'Split per klaster near-duplicate ({len(by_cluster):,} klaster), 70/13/17.', '']
if missing:
    L += [f'**Sumber yang TIDAK dipakai: {", ".join(missing)}.** '
          'Ini wajib disebut bersama angka mana pun di bawah.', '']
L += ['| Metrik | Nilai |', '|---|---|',
      f'| Akurasi (test) | {acc_te:.3f} |',
      f'| Macro-F1 (test) | {mf1:.3f} |',
      f'| ECE sebelum kalibrasi | {e_before:.3f} |',
      f'| ECE sesudah (T={best_t:.2f}) | {e_after:.3f} |',
      f'| Ambang abstain | {TH:.2f} (cakupan {cov[pick]:.0%}) |',
      f'| Model int8 | {size_kb:.0f} KB |',
      f'| Latensi (CPU Kaggle) | {lat:.1f} ms |', '',
      '## Per kelas', '', '| Kelas | P | R | F1 | n |', '|---|---|---|---|---|']
for c in range(N):
    v = prf[c]
    if v['support']:
        L.append(f'| {LABEL_ID[TRAIN_CLASSES[c]]} | {v["precision"]:.3f} | {v["recall"]:.3f} '
                 f'| {v["f1"]:.3f} | {v["support"]} |')
if res:
    L += ['', '## Lintas-dataset (macro-F1)', '', '| Dilatih | Diuji | Macro-F1 |', '|---|---|---|']
    L += [f'| {a} | {b} | {v:.3f} |' for (a, b), v in sorted(res.items())]

L += ['', '## Pemetaan ke enum aplikasi', '',
      'Kelas yang dilatih bukan nilai `MaterialType`. Inilah terjemahannya, dan '
      'inilah yang membuat model dapat dipasang sama sekali:', '',
      '| Kelas latih | MaterialType | Grade |', '|---|---|---|']
for c in TRAIN_CLASSES:
    m_, g_, _ = TO_APP[c]
    L.append(f'| `{c}` | `{m_}` | {f"`{g_}`" if g_ else "—"} |')

L += ['', '## Batas yang wajib dinyatakan', '',
      f'Model mengeluarkan {N} kelas latih yang menjadi '
      f'{len({TO_APP[c][0] for c in TRAIN_CLASSES})} dari 12 `MaterialType`. '
      f'Empat tidak tercapai sama sekali — {", ".join(UNREACHABLE_TYPES)} — dan justru '
      'itulah yang paling sering ditemui pemulung: kresek, gelas plastik, sedotan, '
      'styrofoam.', '',
      f'Dari {ALL_GRADES_N} grade papan harga, hanya {len(REACHABLE_GRADES)} '
      f'({", ".join(REACHABLE_GRADES)}) yang dapat diturunkan tanpa menebak. Pembedaan '
      'yang menentukan selisih harga terbesar — bening versus berwarna, koran versus '
      'duplex, tembaga versus besi — tidak dilabeli dataset mana pun.', '',
      'Model ini karena itu **asisten identifikasi kasar, bukan penentu harga**. '
      'Kode resin tetap jalur utama karena ia fakta, bukan tebakan.', '',
      '## Atribusi dataset', '']
for s in present:
    L.append(f'- **{SOURCE_META[s]["name"]}** ({SOURCE_META[s]["license"]}) — '
             f'{SOURCE_META[s]["attribution"]}')

(OUT / 'report.md').write_text('\n'.join(L))
print('\n'.join(L[:20]))
print(f'\nArtefak tersimpan di {OUT} — unduh lewat tab Output di panel kanan.')
print('Berkas:', sorted(p.name for p in OUT.iterdir()))

## Yang boleh dan tidak boleh diklaim

**Boleh** — macro-F1 dan tabel per kelas dari test set milikmu sendiri; angka
lintas-dataset; ukuran model dan ambang abstain; perbandingan dengan Thung & Yang
karena rasio splitnya sama.

**Tidak boleh** — akurasi dari makalah orang lain; "AI menentukan harga" (harga
datang dari bukti timbang, bukan dari model); latensi Kaggle sebagai latensi
perangkat; grade apa pun di luar yang benar-benar tercapai; dan **angka apa pun
dari mode SMOKE**.

Kalau ada sumber yang tidak dilampirkan, `report.md` sudah mencatatnya. Sebutkan
itu bersama angkanya — bukan karena mengurangi nilai, tetapi karena juri yang
menemukannya sendiri akan mempertanyakan semua angka yang lain.

## Langkah berikutnya

Bagian yang benar-benar baru bukan model ini, melainkan dataset yang belum ada:
**foto material per-grade sebagaimana dipakai lapak Indonesia**. 30–50 foto per
grade sudah cukup untuk mulai, dan pengumpulannya bisa digabung dengan wawancara
pemulung yang memang diminta juri.